# DERMS-R on Colab Pro — BloodMNIST benchmark

Runs this repo's `train.py` against **BloodMNIST** from the MedMNIST v2 collection: 8-class peripheral blood cell classification, ~17k images, already deduplicated and split by MedMNIST's curators. `bloodmnist` is its own first-class entry in `derms/config.py`'s `DATASETS`/`IMAGEFOLDER_DATASETS` — a separate dataset from this repo's other ImageFolder-backed dataset, `all` (the raw Blood Cell Cancer [ALL] data).

- **Repo**: `edramos-lab/derms-r` (public, cloned fresh each session)
- **Tracking**: standalone MLflow, local file store under `--output` (no dependency on the RTX host's MLflow server — that server isn't reachable from Colab's cloud VM over the local LAN)
- **Data**: downloaded directly from MedMNIST's Zenodo-backed source via `prepare_bloodmnist.py`, cached on Drive so later sessions skip the download; converted into this repo's ImageFolder layout at `datasets/bloodmnist/{development,test}` on local disk
- **Resilience**: training writes to local Colab disk and is periodically rsynced to Drive, so a runtime disconnect loses at most a few minutes of progress; `--resume` picks back up

**Before running**: Runtime menu -> Change runtime type -> GPU (Colab Pro: A100/V100/T4 depending on availability).

In [ ]:
!nvidia-smi

## 1. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/derms-r'
MEDMNIST_CACHE = f'{DRIVE_ROOT}/medmnist_cache'  # cached .npz download, reused across sessions
DRIVE_OUTPUT = f'{DRIVE_ROOT}/results'  # synced copy of training results/MLflow store

REPO_DIR = '/content/derms-r'
LOCAL_OUTPUT = '/content/results'  # fast local disk; synced to DRIVE_OUTPUT periodically

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(MEDMNIST_CACHE, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

## 2. Clone the repo and install dependencies

**Prerequisite**: the `bloodmnist` dataset support (`derms/config.py`, `derms/datasets.py`, `derms/engine.py`, `test.py`) must be committed and pushed to `origin/main` first — this cell clones the public GitHub repo fresh, not your local working copy.

Colab's GPU runtime ships a CUDA-matched `torch`/`torchvision` already — we deliberately skip reinstalling those (reinstalling from `requirements.txt`'s generic wheels can silently fall back to a CPU build or a mismatched CUDA version) and only install the rest of the repo's dependencies, plus `medmnist` for the dataset.

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/edramos-lab/derms-r.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!grep -v -E '^(torch|torchvision)([>=<!~ ]|$)' requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt medmnist

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 3. Download BloodMNIST and convert to this repo's ImageFolder layout

`prepare_bloodmnist.py` downloads the official train/val/test `.npz` splits (cached under `MEDMNIST_CACHE` on Drive so repeat runs skip the download), decodes them to PNGs, and writes `datasets/bloodmnist/development` (train+val) and `datasets/bloodmnist/test`. `--data` then points at the generic `datasets` parent, and `train.py --datasets bloodmnist` resolves `<--data>/bloodmnist` to find it. `--size 224` avoids upsampling tiny images later in the pipeline.

Two leakage guards in `derms/engine.py` are relevant here, both by design (never silently drop test samples):
- **Content leakage**: a handful of images are byte-identical across MedMNIST's own official train/val and test splits (observed: 6 out of ~17k) — a data-quality artifact of the source, not this script. `prepare_bloodmnist.py` now hashes test images first and drops any matching development image, logging the count, so this is caught here instead of after a full training run.
- **Patient/group leakage**: when `--all-independent-images` isn't passed, `train.py` falls back to inferring patient/sample groups from filenames (built for the other `all` dataset's `Snap_018 (2).jpg`-style naming, where `(2)` marks a repeat crop of the same sample). BloodMNIST has no such grouping — every image is an independent cell — but this export's sequential `00000.png`, `00001.png`, ... filenames reset per class **per split**, so e.g. `basophil/00000.png` exists in both `development` and `test` and the filename-inference heuristic misreads that as the same "patient" appearing on both sides. All three `train.py` invocations below pass `--all-independent-images` to opt out of that inference entirely, which is the semantically correct choice for this dataset.

In [ ]:
DATA_ROOT = f'{REPO_DIR}/datasets'

!python prepare_bloodmnist.py --out_dir "{DATA_ROOT}/bloodmnist" --size 224 --download_root "{MEDMNIST_CACHE}"

## 4. Verify model weights download correctly

In [ ]:
!python train.py --verify-models

## 5. Background sync to Drive

Runs `rsync` against `LOCAL_OUTPUT` every 5 minutes in a background thread so results/MLflow data survive a disconnect. Leave this cell's thread running for the rest of the session; re-running it is harmless (old thread just keeps looping).

In [ ]:
import threading, time, subprocess

def _sync_loop():
    os.makedirs(LOCAL_OUTPUT, exist_ok=True)
    while True:
        subprocess.run(['rsync', '-a', '--update', f'{LOCAL_OUTPUT}/', f'{DRIVE_OUTPUT}/'],
                       capture_output=True)
        time.sleep(300)

threading.Thread(target=_sync_loop, daemon=True).start()
print('background sync started: LOCAL_OUTPUT -> DRIVE_OUTPUT every 5 min')

## 6. Smoke test

Two lightweight architectures, 2 epochs, before committing to the full run. Confirms the data pipeline, GPU, and MLflow logging all work end to end.

If you already ran this once against uncorrected data (content or group leakage), delete the old output first — every run writes a `splits.json` manifest and refuses to continue if the dataset or flags changed since: `!rm -rf /content/results_smoke`.

In [ ]:
!python train.py \
  --datasets bloodmnist \
  --models mobilenetv4 efficientnet_b0 \
  --epochs 2 \
  --workers 2 \
  --data "{DATA_ROOT}" \
  --output /content/results_smoke \
  --device cuda:0 \
  --all-independent-images \
  --mlflow --mlflow-experiment derms-r-bloodmnist-smoke

## 7. Full benchmark run

`train.py` has no `--ratios` flag — one invocation always runs all 5 ratios x 5 folds internally. With the default 12-architecture registry on the `bloodmnist` dataset that's 12 x 5 x 5 = 300 fits plus the final lock, at `float32` (TF32 and nondeterministic kernels are disabled repo-wide for reproducibility, so this is slower than typical "FP32" benchmarks assume).

This will very likely outlive a single Colab session. `--resume` makes re-running this exact cell safe after a disconnect — completed fits are skipped. Narrow `--models` to a subset first if you want faster turnaround.

In [ ]:
!python train.py \
  --datasets bloodmnist \
  --data "{DATA_ROOT}" \
  --output {LOCAL_OUTPUT} \
  --device cuda:0 \
  --workers 2 \
  --resume \
  --all-independent-images \
  --mlflow --mlflow-experiment derms-r-bloodmnist

## 8. Lock the winning architecture

Once the full CV run above has completed (all folds/ratios/models), select the winner, refit on the full development set, and evaluate on the held-out test set.

In [ ]:
!python train.py \
  --datasets bloodmnist \
  --data "{DATA_ROOT}" \
  --output {LOCAL_OUTPUT} \
  --device cuda:0 \
  --resume --lock bloodmnist \
  --all-independent-images \
  --mlflow --mlflow-experiment derms-r-bloodmnist

## 9. Final sync

The background thread from section 5 syncs every 5 minutes already; run this once more at the end to be sure the last few minutes made it to Drive before you close the runtime.

In [ ]:
!rsync -a --update {LOCAL_OUTPUT}/ "{DRIVE_OUTPUT}/"
print('synced', LOCAL_OUTPUT, '->', DRIVE_OUTPUT)